# 02 - Prompt Engineering & Structured Output

In the previous section, we learned how LLMs work and how to make basic API calls.  

However, getting useful results from LLMs requires more than just asking questions.  
You need to **craft effective prompts** and sometimes **enforce output structure**.

### The Challenge

When working with LLMs, you often face two key problems:

1. **Inconsistent responses** — The same question can produce different formats or levels of detail
2. **Unstructured output** — Free-form text is great for humans, but hard to parse programmatically

### What You'll Learn

- How to write better prompts to guide model behavior
- Techniques like few-shot learning and chain-of-thought reasoning
- How to enforce structured output using JSON schemas
- When to use structured vs unstructured responses

## The Problem: Unreliable Free-Form Text

Let's start by asking an LLM to extract information from text.  
We'll see how inconsistent the output can be without proper prompting.

### The Traditional Approach: String Parsing

Before structured output, developers had to parse free-form LLM responses using regex, string splitting, or brittle heuristics.

Example task: Extract flight booking details from a customer email.

In [ ]:
import re

def parse_flight_details_regex(email_text: str) -> dict:
    """Extract flight details using regex (fragile, hard to maintain)."""
    result = {}
    
    # Try to find departure city
    departure_match = re.search(r"from\s+([A-Z][a-z]+)", email_text)
    if departure_match:
        result["departure"] = departure_match.group(1)
    
    # Try to find destination
    destination_match = re.search(r"to\s+([A-Z][a-z]+)", email_text)
    if destination_match:
        result["destination"] = destination_match.group(1)
    
    # Try to find date (many formats possible!)
    date_match = re.search(r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", email_text)
    if date_match:
        result["date"] = date_match.group(1)
    
    # Try to find passenger count
    passenger_match = re.search(r"(\d+)\s+passengers?", email_text, re.IGNORECASE)
    if passenger_match:
        result["passengers"] = int(passenger_match.group(1))
    
    return result

In [ ]:
# Test emails with different formats
test_emails = [
    "I need to fly from Amsterdam to Paris on 15/06/2024 for 2 passengers",
    "Book me a flight: Amsterdam → Paris, June 15th, two people",  # <-- Different format!
    "I want to travel to Paris from Amsterdam next Friday with my colleague",  # <-- Vague!
]

print("=== Regex-based Extraction ===")
for email in test_emails:
    print(f"\nEmail: '{email}'")
    print(f"Extracted: {parse_flight_details_regex(email)}")
    print("------" * 30)

**Notice the failures:**
- "June 15th" is not matched (expects numeric format)
- "two people" is not recognized (expects digits)
- "next Friday" is completely missed
- The arrow "→" breaks the pattern matching

This is where LLMs + good prompting can help!

## Setup

#### Making the first API call

Before running the below cell, ensure you have:

1. Authenticated with `gcloud auth application-default login`
2. Set your GCP project and location below

The code below creates a genai client configured for Vertex AI.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path().resolve().parents[1] / ".env")

In [ ]:
# This workshop uses the `google-genai` Python package with Vertex AI.
from google import genai

# Set GCP project and location
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_LOCATION")

# Initialize the genai client for Vertex AI
client = genai.Client(
    vertexai=True, 
    project=project_id, 
    location=location
)

For simplicity, we'll define a helper function to call the LLM

In [ ]:
from google.genai.types import GenerateContentConfig

# Select the model
MODEL_NAME = "gemini-2.5-flash"
# Set a default system message for the model
SYSTEM_MESSAGE = "You are a helpful assistant"


def ask_llm(
    prompt: str,
    system_instruction: str = SYSTEM_MESSAGE,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 1.0,
    response_mime_type: str = None,  # <-- NEW: for structured output
    response_schema: dict = None,     # <-- NEW: for structured output
) -> str:
    """Send a prompt to the LLM and return the text response."""
    config = GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        response_mime_type=response_mime_type,  # <-- Controls output format (e.g., "application/json")
        response_schema=response_schema,         # <-- Enforces JSON structure
    )
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )
    return response.text

## Level 1: Basic Prompting

Let's start with a naive prompt and see what we get.

In [ ]:
email = "I need to fly from Amsterdam to Paris on 15/06/2024 for 2 passengers"

# Vague prompt
prompt = f"Extract flight details from this email: {email}"

response = ask_llm(prompt, temperature=0.2)
print(response)

**Problem:** The output format is unpredictable. Sometimes it's a paragraph, sometimes bullet points, sometimes partial.

## Level 2: Improved Prompting

Let's be more specific about what we want and how we want it formatted.

In [ ]:
# Better prompt with clear instructions
prompt = f"""
Extract the following flight details from the email below:
- Departure city
- Destination city
- Travel date
- Number of passengers

Email: {email}

Format your response as:
Departure: [city]
Destination: [city]
Date: [date]
Passengers: [number]
"""

response = ask_llm(prompt, temperature=0.2)
print(response)

**Better!** But still not ideal for programmatic use. We'd need to parse the string.

## Level 3: Few-Shot Learning

Show the model examples of the behavior you want. This technique is called **few-shot prompting**.

In [ ]:
# Few-shot prompt with examples
prompt = f"""
Extract flight details from customer emails.

Example 1:
Email: "I want to go from London to Berlin on March 5th, just me"
Output:
Departure: London
Destination: Berlin
Date: 2024-03-05
Passengers: 1

Example 2:
Email: "Book 3 tickets Barcelona → Rome for next Monday"
Output:
Departure: Barcelona
Destination: Rome
Date: [next Monday's date]
Passengers: 3

Now extract from this email:
Email: {email}
Output:
"""

response = ask_llm(prompt, temperature=0.2)
print(response)

**Even better!** The model now follows the pattern we showed. But we still need to parse text.

## Level 4: Structured Output with JSON

Instead of asking for text we need to parse, we can **enforce a JSON schema**.  
This guarantees the output format and makes it directly usable in code.

In [ ]:
import json

# Define the expected JSON structure
schema = {
    "type": "object",
    "properties": {
        "departure": {"type": "string", "description": "Departure city"},
        "destination": {"type": "string", "description": "Destination city"},
        "date": {"type": "string", "description": "Travel date in YYYY-MM-DD format"},
        "passengers": {"type": "integer", "description": "Number of passengers"},
    },
    "required": ["departure", "destination", "date", "passengers"],
}

prompt = f"Extract flight booking details from this email: {email}"

response = ask_llm(
    prompt,
    temperature=0.2,
    response_mime_type="application/json",  # <-- Enforce JSON
    response_schema=schema,                  # <-- Enforce schema
)

print("Raw response:")
print(response)
print("\nParsed as JSON:")
data = json.loads(response)
print(data)
print(f"\nDeparture city (programmatic access): {data['departure']}")

**Perfect!** Now we get guaranteed JSON that matches our schema. No parsing needed!

## Test Structured Output with Different Emails

Let's see how well the structured approach handles various input formats.

In [ ]:
test_emails = [
    "I need to fly from Amsterdam to Paris on 15/06/2024 for 2 passengers",
    "Book me a flight: Amsterdam → Paris, June 15th, two people",
    "I want to travel to Paris from Amsterdam next Friday with my colleague",
    "3 tickets from London to New York on Christmas Day please",
]

print("=== Structured Output Extraction ===")
for email in test_emails:
    prompt = f"Extract flight booking details from this email: {email}"
    response = ask_llm(
        prompt,
        temperature=0.2,
        response_mime_type="application/json",
        response_schema=schema,
    )
    data = json.loads(response)
    print(f"\nEmail: '{email}'")
    print(f"Extracted: {data}")
    print("------" * 30)

## Chain-of-Thought Reasoning

For complex tasks, we can ask the model to **show its reasoning** before giving an answer.  
This technique is called **Chain-of-Thought (CoT)** prompting.

Example: Determine if a flight booking request is feasible.

In [ ]:
# Complex scenario
booking_request = """
I need to book a flight from Amsterdam to Sydney, departing tomorrow morning.
I have a budget of 200 euros and need to arrive by tomorrow evening.
I also need to bring my 3 large suitcases.
"""

# Without chain-of-thought
prompt_simple = f"""
Is this flight booking request feasible? Answer yes or no.

Request: {booking_request}
"""

print("=== Without Chain-of-Thought ===")
response = ask_llm(prompt_simple, temperature=0.2)
print(response)

In [ ]:
# With chain-of-thought
prompt_cot = f"""
Analyze if this flight booking request is feasible.

Request: {booking_request}

Think step by step:
1. What is the typical flight duration from Amsterdam to Sydney?
2. Is it possible to arrive by tomorrow evening if departing tomorrow morning?
3. What is a typical budget for this route?
4. Are there typically baggage restrictions?

After reasoning through these points, conclude with "Feasible: Yes" or "Feasible: No".
"""

print("\n=== With Chain-of-Thought ===")
response = ask_llm(prompt_cot, temperature=0.2)
print(response)

**Notice:** With CoT, the model reasons through the problem and provides justification. More reliable for complex decisions!

## Combining CoT with Structured Output

We can get both reasoning AND structured output by designing the right schema.

In [ ]:
# Schema that includes reasoning
feasibility_schema = {
    "type": "object",
    "properties": {
        "flight_duration_hours": {"type": "number"},
        "can_arrive_on_time": {"type": "boolean"},
        "budget_sufficient": {"type": "boolean"},
        "baggage_feasible": {"type": "boolean"},
        "reasoning": {"type": "string"},
        "feasible": {"type": "boolean"},
    },
    "required": ["flight_duration_hours", "can_arrive_on_time", "budget_sufficient", 
                 "baggage_feasible", "reasoning", "feasible"],
}

prompt = f"""
Analyze if this flight booking request is feasible.

Request: {booking_request}

Consider:
- Flight duration from Amsterdam to Sydney
- Whether arrival by tomorrow evening is possible
- Whether 200 euros is a realistic budget
- Baggage restrictions for 3 large suitcases
"""

response = ask_llm(
    prompt,
    temperature=0.2,
    response_mime_type="application/json",
    response_schema=feasibility_schema,
)

data = json.loads(response)
print("Structured Feasibility Analysis:")
print(json.dumps(data, indent=2))

## Experiment: Try Different System Instructions

The **system instruction** sets the model's behavior globally for the conversation.  
Try changing the personality and see how it affects responses.

In [ ]:
email = "I need to fly from Amsterdam to Paris on 15/06/2024 for 2 passengers"
prompt = f"Extract flight booking details from this email: {email}"

# Professional assistant
print("=== Professional Assistant ===")
response = ask_llm(
    prompt,
    system_instruction="You are a professional travel booking assistant. Be concise and accurate.",
    temperature=0.2,
    response_mime_type="application/json",
    response_schema=schema,
)
print(json.loads(response))

print("\n=== Cautious Assistant ===")
# Cautious assistant
response = ask_llm(
    prompt,
    system_instruction="You are a cautious assistant. If any detail is unclear or missing, indicate it in the output.",
    temperature=0.2,
)
print(response)

## Exercise: Build Your Own Structured Extractor

Now it's your turn! Create a structured output extractor for a different domain.

**Task:** Extract product review information from customer feedback.

Your schema should include:
- `product_name` (string)
- `rating` (integer, 1-5)
- `sentiment` (string: "positive", "negative", or "neutral")
- `key_points` (array of strings)
- `would_recommend` (boolean)

Test it on these reviews:
1. "I love my new headphones! The sound quality is amazing and they're super comfortable. 5 stars!"
2. "The laptop is okay but overpriced. Battery life is disappointing. Maybe 3/5."
3. "Terrible experience with this phone. Screen cracked after 2 days. Would not recommend."

In [ ]:
# Define your schema here
review_schema = {
    "type": "object",
    "properties": {
        # TODO: Add your properties
    },
    "required": [],  # TODO: Add required fields
}

test_reviews = [
    "I love my new headphones! The sound quality is amazing and they're super comfortable. 5 stars!",
    "The laptop is okay but overpriced. Battery life is disappointing. Maybe 3/5.",
    "Terrible experience with this phone. Screen cracked after 2 days. Would not recommend.",
]

# TODO: Write code to extract review data using structured output
# for review in test_reviews:
#     prompt = ...
#     response = ask_llm(...)
#     ...

## Exercise: Implement Chain-of-Thought for Complex Classification

Create a customer support ticket classifier that uses chain-of-thought reasoning.

**Task:** Classify support tickets into urgency levels (low, medium, high, critical) with reasoning.

The model should:
1. Identify key indicators (keywords, sentiment, business impact)
2. Reason through the urgency level
3. Provide a final classification with confidence score

Test tickets:
1. "My app keeps crashing when I try to make a payment. This is urgent!"
2. "Can you add a dark mode feature? It would be nice to have."
3. "Our entire production system is down. All customers are affected. Need immediate help!"
4. "I forgot my password. Can you help me reset it?"

In [ ]:
# Define your schema for ticket classification
ticket_schema = {
    "type": "object",
    "properties": {
        # TODO: Design schema including reasoning, urgency, confidence, etc.
    },
    "required": [],
}

test_tickets = [
    "My app keeps crashing when I try to make a payment. This is urgent!",
    "Can you add a dark mode feature? It would be nice to have.",
    "Our entire production system is down. All customers are affected. Need immediate help!",
    "I forgot my password. Can you help me reset it?",
]

# TODO: Implement chain-of-thought classification

## Key Takeaways

1. **Better prompts = better results**  
   Be specific, provide examples, and guide the model's reasoning.

2. **Few-shot learning**  
   Show examples of desired behavior to teach the model patterns.

3. **Chain-of-thought**  
   Ask the model to reason step-by-step for complex tasks.

4. **Structured output**  
   Use JSON schemas to guarantee format and eliminate parsing errors.

5. **System instructions matter**  
   Set the right persona and guidelines for consistent behavior.

### When to Use What

| Use Case | Technique |
|----------|----------|
| Need exact format for downstream processing | Structured output with schema |
| Complex reasoning or multi-step logic | Chain-of-thought prompting |
| Model doesn't understand task | Few-shot examples |
| Inconsistent responses | Lower temperature + clearer instructions |
| Need to parse LLM output | Use structured output instead! |

## Bonus: Try Your Own Prompts

Experiment with different prompting techniques on your own use cases!

In [ ]:
# Free experimentation cell
# Try your own prompts here!